# Artificial Structural Conditioning Analysis

This notebook measures whether structural conditioning steers generated artificial graphs toward requested path, cycle, and ray statistics, using compact tables, plots, and graph examples.

This notebook is intentionally thin: dataset loading, graph diagnostics, and plotting live in reusable package helpers. It checks whether conditioning on a structural statistic steers generated artificial graphs toward that statistic.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
from IPython.display import display

from conditional_node_field_graph_generator.notebooks import configure_notebook
from conditional_node_field_graph_generator.extensions.demo.artificial import (
    build_artificial_plotter,
    find_latest_artificial_dataset_config,
    load_latest_artificial_graph_generator,
    generate_artificial_dataset,
)
from conditional_node_field_graph_generator.extensions.demo.artificial_conditioning import (
    histogram_tables,
    plot_conditioning_vector_test,
    run_conditioning_vector_test,
)

globals().update(configure_notebook(require_nsppk=True, print_torch=True))

In [ ]:
dataset_config_path = find_latest_artificial_dataset_config(REPO_ROOT / 'notebooks' / 'configs' / 'artificial_datasets')
graphs, _ = generate_artificial_dataset(load_from_file=dataset_config_path, save_config=False)
graph_generator, generator_path = load_latest_artificial_graph_generator()
plotter = build_artificial_plotter()

print(f'Dataset: {dataset_config_path.name} ({len(graphs)} graphs)')
print(f'Generator: {Path(generator_path).name}')
plotter(graphs[:20], n_cols=min(5, len(graphs[:20])), titles=[f'graph {i}' for i in range(len(graphs[:20]))])

In [ ]:
sample_variants = graph_generator.sample(3, return_decode_stages=True)
for variant_key, title_prefix in (('raw', 'raw'), ('ilp', 'ilp'), ('oracle', 'oracle ilp')):
    variant_samples = sample_variants[variant_key]
    plotter(
        variant_samples,
        n_cols=len(variant_samples),
        titles=[f'{title_prefix} {idx}' for idx in range(len(variant_samples))],
    )

## Conditioning test

Change `experiment_type` to `cycle_count`, `cycle_size`, `ray_count`, or `ray_size` to probe another structural statistic.

In [ ]:
result = run_conditioning_vector_test(
    graphs,
    graph_generator,
    experiment_type='path_length',
    samples_per_conditioning_graph=32,
    feasibility_effort=0,
)

display(result['summary_df'])
plot_conditioning_vector_test(result, plotter, examples_per_condition=4)

In [ ]:
condition_index = min(2, len(result['conditioning_graphs']) - 1)
label_table, degree_table = histogram_tables(
    result['conditioning_graphs'][condition_index],
    result['generated_by_graph'][condition_index],
)
display(label_table)
display(degree_table)